# Pricing Barrier Options

This notebook shows how to price single-barrier options using `derivatives_pricing`'s four
engines - analytical, binomial, finite differences and Monte Carlo. It covers knock-ins and 
knock-outs, continuous and discrete monitoring, rebates, American exercise, greeks, and discrete dividends.

## 1) Setup

As with every pricing flow:

```
DiscountCurve → MarketData → UnderlyingData → OptionValuation
```

In [ ]:
import datetime as dt
from dataclasses import replace as dc_replace

import derivatives_pricing as dp

In [ ]:
pricing_date = dt.datetime(2025, 1, 1)
maturity = dt.datetime(2026, 1, 1)  # 1-year option

curve = dp.DiscountCurve.flat(rate=0.05, end_time=2.0)
market = dp.MarketData(
    pricing_date=pricing_date,
    discount_curve=curve,
    currency="USD",
)
underlying = dp.UnderlyingData(
    initial_value=100.0,
    volatility=0.25,
    market_data=market,
)

## 2) Down-and-Out Call

Down-and-out call (DOC) with strike 100, barrier at 85, continuous
monitoring.  Using the analytical engine (`PricingMethod.BSM`).

In [ ]:
doc_spec = dp.BarrierSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
    barrier=85.0,
    direction=dp.BarrierDirection.DOWN,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)

opt_bsm = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.BSM)
print(f"DOC BSM price: {opt_bsm.present_value():.4f}")

DOC BSM price: 11.0529


## 3) Engine Comparison

All three deterministic barrier-capable engines price the same contract.
For continuous monitoring BSM uses the Reiner-Rubinstein closed form,
Binomial uses a Cox-Ross-Rubinstein tree with Boyle-Lau step alignment,
and PDE_FD builds a grid truncated at the barrier.

In [ ]:
opt_bn = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.BINOMIAL)
opt_fd = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.PDE_FD)

print(f"{'Method':<12} {'Price':>10}")
print("-" * 24)
print(f"{'Analytical':<12} {opt_bsm.present_value():>10.4f}")
print(f"{'Binomial':<12} {opt_bn.present_value():>10.4f}")
print(f"{'PDE_FD':<12} {opt_fd.present_value():>10.4f}")

Method            Price
------------------------
Analytical      11.0529
Binomial        11.0558
PDE_FD          11.0529


## 4) In/Out Parity

For European barriers without rebate, the knock-in price + knock-out price
equals the vanilla price.  This is a useful sanity check and also what `derivatives_pricing`
uses internally to implement European knock-in pricing in PDE_FD and BSM.

In [ ]:
dic_spec = dc_replace(doc_spec, action=dp.BarrierAction.IN)

vanilla_spec = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)
pv_doc = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.BSM).present_value()
pv_dic = dp.OptionValuation(underlying, dic_spec, dp.PricingMethod.BSM).present_value()
pv_van = dp.OptionValuation(underlying, vanilla_spec, dp.PricingMethod.BSM).present_value()

print(f"DOC + DIC = {pv_doc + pv_dic:.4f}")
print(f"Vanilla   = {pv_van:.4f}")
print(f"Diff      = {pv_doc + pv_dic - pv_van:+.2e}")

DOC + DIC = 12.3360
Vanilla   = 12.3360
Diff      = -7.11e-15


## 5) All Four Barrier Types

Switching barrier direction/action is a one-field change via
`dataclasses.replace`.  For up barriers we move the barrier above spot.

In [ ]:
up_spec = dc_replace(doc_spec, direction=dp.BarrierDirection.UP, barrier=120.0)

print(f"{'Type':<18} {'Price':>10}")
print("-" * 30)
for direction, action, label in [
    (dp.BarrierDirection.DOWN, dp.BarrierAction.OUT, "down-and-out"),
    (dp.BarrierDirection.DOWN, dp.BarrierAction.IN, "down-and-in"),
    (dp.BarrierDirection.UP, dp.BarrierAction.OUT, "up-and-out"),
    (dp.BarrierDirection.UP, dp.BarrierAction.IN, "up-and-in"),
]:
    barrier = 85.0 if direction is dp.BarrierDirection.DOWN else 120.0
    spec = dc_replace(doc_spec, direction=direction, action=action, barrier=barrier)
    pv = dp.OptionValuation(underlying, spec, dp.PricingMethod.BSM).present_value()
    print(f"{label:<18} {pv:>10.4f}")

Type                    Price
------------------------------
down-and-out          11.0529
down-and-in            1.2831
up-and-out             0.6913
up-and-in             11.6447


## 6) Discrete Monitoring

One can specify discrete monitoring dates - either using
`num_observations` (equally spaced dates from `T/N` to `T`, excluding the
pricing date) or `monitoring_dates` (an explicit schedule) on `BarrierSpec`

BSM uses the Broadie-Glasserman-Kou continuity correction for discrete
barriers. Binomial snaps each monitoring date to its nearest tree layer; PDE_FD
inserts the monitoring dates directly into the time grid, so the barrier
check falls on exact nodes.

In [ ]:
doc_discrete = dc_replace(
    doc_spec,
    monitoring=dp.BarrierMonitoring.DISCRETE,
    num_observations=50,
)

print(f"{'Method':<12} {'Cont.':>10} {'Discrete (m=50)':>18}")
print("-" * 42)
for method in (
    dp.PricingMethod.BSM,
    dp.PricingMethod.BINOMIAL,
    dp.PricingMethod.PDE_FD,
):
    pv_c = dp.OptionValuation(underlying, doc_spec, method).present_value()
    pv_d = dp.OptionValuation(underlying, doc_discrete, method).present_value()
    print(f"{method.value:<12} {pv_c:>10.4f} {pv_d:>18.4f}")

Method            Cont.    Discrete (m=50)
------------------------------------------
bsm             11.0529            11.4489
binomial        11.0558            11.3924
pde_fd          11.0529            11.4491


An explicit monitoring schedule works as follows:

In [ ]:
custom_dates = [pricing_date + dt.timedelta(days=d) for d in (30, 60, 120, 180, 270)]
doc_custom_dates = dc_replace(
    doc_spec,
    monitoring=dp.BarrierMonitoring.DISCRETE,
    monitoring_dates=custom_dates,
)
pv_custom = dp.OptionValuation(
    underlying, doc_custom_dates, dp.PricingMethod.PDE_FD
).present_value()
print(f"DOC with 5 custom monitoring dates: {pv_custom:.4f}")

DOC with 5 custom monitoring dates: 11.8989


## 7) Rebates

A rebate is a fixed cash payment made when the barrier event occurs
(knock-out extinguishes the option but pays the rebate; knock-in pays the
rebate at expiry if the option never activates).

For knock-out barriers the rebate can be paid `AT_HIT` (immediately when
the barrier is hit) or `AT_EXPIRY` (discounted back from maturity).  For
knock-in barriers the rebate is always paid at expiry.

In [ ]:
doc_with_rebate = dc_replace(doc_spec, rebate=2.0, rebate_timing=dp.RebateTiming.AT_HIT)

pv_no_rebate = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.BSM).present_value()
pv_rebate_at_hit = dp.OptionValuation(
    underlying, doc_with_rebate, dp.PricingMethod.BSM
).present_value()
pv_rebate_at_expiry = dp.OptionValuation(
    underlying,
    dc_replace(doc_with_rebate, rebate_timing=dp.RebateTiming.AT_EXPIRY),
    dp.PricingMethod.BSM,
).present_value()

print(f"DOC no rebate:             {pv_no_rebate:.4f}")
print(f"DOC + $2 rebate at hit:    {pv_rebate_at_hit:.4f}")
print(f"DOC + $2 rebate at expiry: {pv_rebate_at_expiry:.4f}")

DOC no rebate:             11.0529
DOC + $2 rebate at hit:    12.0151
DOC + $2 rebate at expiry: 11.9862


## 8) American Exercise

Below we price an American UP and OUT Put option using the Binomial and PDE_FD engines

In [ ]:
uop_spec = dp.BarrierSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=115.0,  # deep-ITM put (spot=100)
    maturity=maturity,
    barrier=130.0,  # up-barrier above spot
    direction=dp.BarrierDirection.UP,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)
uop_american = dc_replace(uop_spec, exercise_type=dp.ExerciseType.AMERICAN)

pv_bn_eu = dp.OptionValuation(underlying, uop_spec, dp.PricingMethod.BINOMIAL).present_value()

pv_bn_am = dp.OptionValuation(underlying, uop_american, dp.PricingMethod.BINOMIAL).present_value()

pv_fd_eu = dp.OptionValuation(underlying, uop_spec, dp.PricingMethod.PDE_FD).present_value()

pv_fd_am = dp.OptionValuation(underlying, uop_american, dp.PricingMethod.PDE_FD).present_value()

print(f"UOP European (Binomial): {pv_bn_eu:.4f}")
print(f"UOP European (PDE_FD):   {pv_fd_eu:.4f}")
print(f"UOP American (Binomial): {pv_bn_am:.4f}")
print(f"UOP American (PDE_FD):   {pv_fd_am:.4f}")
print(f"Early-exercise premium (Binomial):  {pv_bn_am - pv_bn_eu:+.4f}")
print(f"Early-exercise premium (PDE_FD):    {pv_fd_am - pv_fd_eu:+.4f}")

UOP European (Binomial): 15.1171
UOP European (PDE_FD):   15.1147
UOP American (Binomial): 16.6296
UOP American (PDE_FD):   16.6280
Early-exercise premium (Binomial):  +1.5125
Early-exercise premium (PDE_FD):    +1.5133


## 9) Greeks

Greeks can be obtained by calling the relevant greek method on each OptionValuation instance. 
The analytical engine currently only supports numerical bump and revalue greeks for barrier options. 
Binomial and PDE_FD provide tree and grid-native delta, gamma and theta. Vega and rho fall 
back to numerical bump-and-revalue.

In [ ]:
def _safe(ov, accessor):
    """Call an OV accessor; return NaN if the call raises."""
    try:
        return getattr(ov, accessor)()
    except dp.DerivativesPricingError:
        return float("nan")

In [ ]:
# European
opt_bsm = dp.OptionValuation(underlying, uop_spec, dp.PricingMethod.BSM)
opt_bn = dp.OptionValuation(underlying, uop_spec, dp.PricingMethod.BINOMIAL)
opt_fd = dp.OptionValuation(underlying, uop_spec, dp.PricingMethod.PDE_FD)

print(f"{'Greek':<8} {'BSM':>10} {'Binomial':>10} {'PDE_FD':>10}")
print("-" * 42)
for greek in ("delta", "gamma", "vega", "theta", "rho"):
    val_bsm = _safe(opt_bsm, greek)
    val_bn = _safe(opt_bn, greek)
    val_fd = _safe(opt_fd, greek)
    print(f"{greek:<8} {val_bsm:>10.4f} {val_bn:>10.4f} {val_fd:>10.4f}")

# Binomial vega raises an exception because bumping vega changes num_steps due to Boyle-Lau step adjustment
# - different tree topologies give noisy vegas

Greek           BSM   Binomial     PDE_FD
------------------------------------------
delta       -0.6501    -0.6501    -0.6501
gamma        0.0119     0.0119     0.0119
vega         0.2715        nan     0.2715
theta        0.0008     0.0008     0.0008
rho         -0.7372    -0.7373    -0.7372


In [ ]:
# American greeks

opt_bn_am = dp.OptionValuation(underlying, uop_american, dp.PricingMethod.BINOMIAL)

opt_fd_am = dp.OptionValuation(underlying, uop_american, dp.PricingMethod.PDE_FD)
print(f"{'Greek':<8} {'Binomial':>10} {'PDE_FD':>10}")
print("-" * 42)
for greek in ("delta", "gamma", "vega", "theta", "rho"):
    val_bn = _safe(opt_bn_am, greek)
    val_fd = _safe(opt_fd_am, greek)
    print(f"{greek:<8} {val_bn:>10.4f} {val_fd:>10.4f}")

Greek      Binomial     PDE_FD
------------------------------------------
delta       -0.7473    -0.7475
gamma        0.0175     0.0175
vega            nan     0.2098
theta       -0.0025    -0.0025
rho         -0.3454    -0.3450


The binomial and finite difference implementation engines cache Greeks - the first call pays
for the solve, subsequent calls to `present_value`, `delta`, `gamma`,
`theta` are O(1) lookups.

present_value and Greeks are also cached at the OptionValuation level.

## 10) Discrete Dividends

Barrier options with discrete cash dividends are supported by PDE_FD.

In [ ]:
underlying_with_divs = dp.UnderlyingData(
    initial_value=100.0,
    volatility=0.25,
    market_data=market,
    discrete_dividends=[
        (dt.datetime(2025, 4, 1), 1.5),
        (dt.datetime(2025, 10, 1), 1.5),
    ],
)

pv_fd_divs = dp.OptionValuation(
    underlying_with_divs, doc_spec, dp.PricingMethod.PDE_FD
).present_value()
print(f"DOC with discrete divs (PDE_FD): {pv_fd_divs:.4f}")
print(f"DOC without divs    (PDE_FD):    {opt_fd.present_value():.4f}")

DOC with discrete divs (PDE_FD): 9.5619
DOC without divs    (PDE_FD):    15.1147


## 11) Monte Carlo

Monte Carlo prices barrier options by simulating GBM paths, checking the
barrier at every monitoring date (or every time step with brownian bridge adjustment, 
for continuous monitoring), and averaging the discounted payoffs.  For MC the
``underlying`` argument is a ``GBMProcess`` (carrying the path simulation
config) rather than an ``UnderlyingData``.

In [ ]:
sim_config = dp.SimulationConfig(paths=100_000, end_date=maturity, num_steps=52)
gbm = dp.GBMProcess(
    market_data=market,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.25),
    sim_config=sim_config,
)

opt_mc = dp.OptionValuation(
    underlying=gbm,
    spec=doc_spec,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)

opt_bsm = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.BSM)
opt_fd = dp.OptionValuation(underlying, doc_spec, dp.PricingMethod.PDE_FD)

print(f"{'Method':<12} {'Price':>10}")
print("-" * 24)
print(f"{'Analytical':<12} {opt_bsm.present_value():>10.4f}")
print(f"{'PDE_FD':<12} {opt_fd.present_value():>10.4f}")
print(f"{'Monte Carlo':<12} {opt_mc.present_value():>10.4f}")

Method            Price
------------------------
Analytical      11.0529
PDE_FD          11.0529
Monte Carlo     11.0759


MC also supports American exercise and discrete dividends.  Below we price an up and out put, 
American exercise, discrete divs.

In [ ]:
uop_american = dp.BarrierSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.AMERICAN,
    strike=110.0,
    maturity=maturity,
    barrier=120.0,
    direction=dp.BarrierDirection.UP,
    action=dp.BarrierAction.OUT,
    monitoring=dp.BarrierMonitoring.CONTINUOUS,
)

divs = [
    (dt.datetime(2025, 4, 1), 1.5),
    (dt.datetime(2025, 10, 1), 1.5),
]
sim_config = dp.SimulationConfig(paths=100_000, end_date=maturity, num_steps=52)
gbm = dp.GBMProcess(
    market_data=market,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.25, discrete_dividends=divs),
    sim_config=sim_config,
)

opt_mc = dp.OptionValuation(
    underlying=gbm,
    spec=uop_american,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)

opt_fd = dp.OptionValuation(dp.as_underlying_data(gbm), uop_american, dp.PricingMethod.PDE_FD)

print(f"UOP American with divs (Monte Carlo): {opt_mc.present_value():.4f}")
print(f"UOP American with divs (PDE_FD):      {opt_fd.present_value():.4f}")

UOP American with divs (Monte Carlo): 13.3380
UOP American with divs (PDE_FD):      13.3273
